In [ ]:
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.schema.runnable import RunnableSequence , RunnableLambda , RunnableParallel , RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
import os


load_dotenv()

llm = HuggingFaceEndpoint(
    repo_id = "google/gemma-2-2b-it",
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_API_KEY"),
    task = "text-generation"
)

model = ChatHuggingFace(llm = llm)

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()
transcript = ytt_api.fetch("qjPH9njnaVU")

print(transcript)

# https://www.youtube.com/watch?v=qjPH9njnaVU

transcript_text = ""
transcript_text = "".join(snippet.text for snippet in transcript)
    

In [ ]:
print(transcript_text)
print(type(transcript_text))

In [ ]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splitter.create_documents([transcript_text])
# create documents expect the string in a list , it can be many string linke ["cwe" , "wfefw" , etc] or one big string like in this case ["cce"]
# should always be in a list , will return each chunk as a Document object which are needed in vector stores.
# if we just do splitter.create_documents(text) , if we do this without wrapping the string in a list , it will be split characterwise , TRY IT!!!

In [ ]:
chunks

In [ ]:
chunks[0].page_content

In [ ]:
from langchain_huggingface import HuggingFaceEndpointEmbeddings


embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
hf = HuggingFaceEndpointEmbeddings(
    model=embedding_model,
    task="feature-extraction",
    huggingfacehub_api_token= os.getenv("HUGGINGFACEHUB_API_KEY"),
)

In [ ]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, hf) 
# chunks are converted to embeddings and stored in vector store

In [ ]:
vector_store.index_to_docstore_id
# every chunk embedding has its own id now

In [ ]:
retriever = vector_store.as_retriever(search_type = 'similarity' , search_kwargs = {"k" : 5})

In [ ]:
retriever.invoke('who is pavel durov')

In [ ]:
prompt = PromptTemplate(
    template="""
You are a helpful assistant.
Answer only from the provided transcript context.
If the context is insufficient , just say you don't know.

{context}
Question : {question}
""",
input_variables=['context', 'question']
)

In [ ]:
question = 'Tell me about Romanian elections'
context = retriever.invoke(question)

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in context)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({'context' : context_text , "question" : question})

In [ ]:
final_prompt

In [ ]:
result = model.invoke(final_prompt)

In [ ]:
result.content

BUILDING A CHAIN

In [ ]:
parser = StrOutputParser()
chain = prompt | model | parser

chain.invoke({'context' : context_text , "question" : question})

# this is a simple chain for our output we are getting , prompt needs 2 inputs , context and question , question is easy we can get it from the user , lets think about context , how to make it via a chain

In [ ]:
# we have 2 chains , one is QUESTION -> QUESTION  
# another chain is QUESTION --> RETRIEVER --> CONTEXT
# lets make both of them

In [ ]:
def format_docs(context):
    context_text = "\n\n".join(doc.page_content for doc in context)
    return context_text

In [ ]:
parallel_chain = RunnableParallel({

    "context" : retriever | RunnableLambda(format_docs),
    "question" : RunnablePassthrough(),
}
)

# first chain gets question as input from the user , therefore chain1 = question | retriever ( retriever outputs docs from vector store , now we need to proess those docs ) | RunnableLambda(format_docs) ---> as format docs is a function we will convert it to a runnable using runnableLambda
# second chain is user gives a input and the same question is the output , therefore RunnablePassthrough()

In [ ]:
# If you want to give different inputs to each chain in RunnableParallel, you don’t pass a single string directly. Instead, you pass a dictionary (or mapping) where the keys match your RunnableParallel keys.

# Here’s how it works:

# Example:
# from langchain.core.runnables import RunnableParallel

# # Define your chains
# chain_a = ...  # some chain
# chain_b = ...  # another chain

# # Create RunnableParallel
# parallel = RunnableParallel(
#     A=chain_a,
#     B=chain_b
# )

# # Pass different inputs as a dictionary
# inputs = {
#     "A": "Input for chain A",
#     "B": "Input for chain B"
# }

# result = parallel.invoke(inputs)

# print(result)
# # Output will be:
# # {
# #    "A": <chain_a output>,
# #    "B": <chain_b output>
# # }
# Key points:
# The keys in the input dictionary must match the keys you defined in RunnableParallel.

# Each chain then only sees its corresponding input.

# This way, RunnableParallel acts like a map of chains to their own inputs, instead of sending the same input to all.

# If you want, I can also show a hybrid example where some chains share the same input and others get different inputs, all in a single RunnableParallel. This is often very useful in multi-task pipelines. Do you want me to show that?

# You said:
# yes show me
# ChatGPT said:
# Perfect! Let’s make a hybrid RunnableParallel example where:

# Chain A and Chain B get different inputs

# Chain C gets the same input as Chain A (or any shared input)

# Example Code
# from langchain.core.runnables import RunnableParallel

# # Define chains (can be LLMChain, RunnableLambda, or other runnables)
# chain_a = ...  # some chain or LLM
# chain_b = ...  # another chain or LLM
# chain_c = ...  # another chain

# # Create RunnableParallel
# parallel = RunnableParallel(
#     A=chain_a,
#     B=chain_b,
#     C=chain_c
# )

# # Prepare inputs
# # - "A" gets its own input
# # - "B" gets a different input
# # - "C" shares input with "A"
# inputs = {
#     "A": "Input specific to chain A",
#     "B": "Input specific to chain B",
#     "C": "Input specific to chain A"  # or any shared value
# }

# # Invoke the parallel runnable
# result = parallel.invoke(inputs)

# print(result)
# # Example output:
# # {
# #     "A": <output of chain A>,
# #     "B": <output of chain B>,
# #     "C": <output of chain C>
# # }
# How it works:
# RunnableParallel looks at the keys in the dictionary you pass to invoke().

# Each chain gets exactly the input mapped to its key.

# Chains run in parallel, so no chain blocks another.

# Output is a dictionary matching the keys, so you can easily access each chain’s output.

In [ ]:
result = parallel_chain.invoke("Tell me about romanian elections")
# this one input goes to both chains in RunnableParallel

In [ ]:
result

In [ ]:
final_chain = parallel_chain | chain

In [ ]:
result = final_chain.invoke("Tell me about romanian elections")

In [ ]:
result

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [ ]:
store = {}

def get_session_history(session_id : str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id] 